# RNN Text Classification

Classify news articles from the **20 Newsgroups** dataset using a plain RNN.

This notebook covers:
- Simple tokeniser + vocabulary
- Fixed-length padding / truncation
- `nn.Embedding` + `nn.RNN`
- Last hidden state as sentence representation

> **Next:** `02_lstm_text_classification.ipynb` upgrades to LSTM/GRU/BiLSTM with variable-length batching (`pack_padded_sequence`) and gradient clipping.

In [ ]:
import re
import numpy as np
import torch
import torch.nn as nn
from sklearn.datasets import fetch_20newsgroups
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset

DEVICE = 'cuda' if torch.cuda.is_available() else \
         'mps'  if torch.backends.mps.is_available() else 'cpu'
print('device:', DEVICE)

## 1. Load Data

In [ ]:
newsgroups = fetch_20newsgroups(subset='all', remove=('headers', 'footers', 'quotes'))
texts      = newsgroups.data
labels     = newsgroups.target
num_classes = len(newsgroups.target_names)
print(f'{len(texts)} docs  {num_classes} classes')
print('Classes:', newsgroups.target_names[:5], '...')

## 2. Vocabulary and Preprocessing

In [ ]:
def tokenize(text):
    return re.sub(r'[^a-z0-9\s]', '', text.lower()).split()

vocab = {'<PAD>': 0, '<UNK>': 1}
for text in texts:
    for tok in tokenize(text):
        if tok not in vocab:
            vocab[tok] = len(vocab)

print(f'Vocab size: {len(vocab):,}')

In [ ]:
MAX_LEN = 100

def text_to_sequence(text):
    seq = [vocab.get(tok, 1) for tok in tokenize(text)]
    # pad or truncate to MAX_LEN
    seq = seq[:MAX_LEN] + [0] * max(0, MAX_LEN - len(seq))
    return seq

sequences = np.array([text_to_sequence(t) for t in texts], dtype=np.int64)
labels    = np.array(labels, dtype=np.int64)
print('sequences shape:', sequences.shape)

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(sequences, labels, test_size=0.2, random_state=42)

train_loader = DataLoader(
    TensorDataset(torch.tensor(X_tr), torch.tensor(y_tr)),
    batch_size=64, shuffle=True
)
print(f'Train: {len(X_tr)}  Test: {len(X_te)}')

## 3. Model

```
token ids  →  Embedding  →  RNN  →  last hidden state  →  Linear  →  logits
[B, T]        [B,T,E]       [B,T,H]  [B, H]               [B, C]
```

We take `out[:, -1, :]` — the RNN output at the final time step — as the sentence representation.

In [ ]:
class RNNTextClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_size, num_layers, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.rnn = nn.RNN(embed_dim, hidden_size, num_layers, batch_first=True)
        self.fc  = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        x   = self.embedding(x)   # [B, T, E]
        h0  = torch.zeros(self.rnn.num_layers, x.size(0),
                          self.rnn.hidden_size).to(x.device)
        out, _ = self.rnn(x, h0)  # [B, T, H]
        return self.fc(out[:, -1, :])  # last step → [B, C]


model = RNNTextClassifier(
    vocab_size=len(vocab), embed_dim=64,
    hidden_size=128, num_layers=1,
    num_classes=num_classes
).to(DEVICE)
print(model)
print(f'Params: {sum(p.numel() for p in model.parameters()):,}')

## 4. Train

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(5):
    model.train()
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        criterion(model(xb), yb).backward()
        optimizer.step()
    print(f'Epoch {epoch+1}/5 done')

## 5. Evaluate

In [ ]:
model.eval()
with torch.no_grad():
    logits = model(torch.tensor(X_te).to(DEVICE))
    acc = (logits.argmax(1).cpu() == torch.tensor(y_te)).float().mean()
print(f'Test accuracy: {acc.item()*100:.2f}%')

## Limitations of plain RNN

| Issue | Why | Fix |
|---|---|---|
| Vanishing gradients | Error signal shrinks over long sequences | LSTM / GRU (gating) |
| Fixed-length padding wastes compute | PAD tokens still processed | `pack_padded_sequence` |
| Unidirectional | Only sees left context | Bidirectional RNN |

→ `02_lstm_text_classification.ipynb` fixes all three.